# Radon transform and CT reconstruction

*medical · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/radon_ct)

A CT scanner records line integrals $p(\theta, s)$ of the attenuation map from every angle. The Shepp–Logan head phantom is projected by rotating and summing a tensor, and rebuilt by filtered back-projection: a ramp filter $|\nu|$ in Fourier space, then one einsum over the angles.

**Vector form**

$$p(\theta, s) = \int f(s\,\hat{\mathbf n}_\theta + t\,\hat{\mathbf n}_\theta^{\perp})\,dt \qquad f(\mathbf r) = \int_0^{\pi} \big(\mathcal F^{-1}|\nu|\,\mathcal F p\big)(\theta,\, \mathbf r\cdot\hat{\mathbf n}_\theta)\,d\theta$$

**Index (tensor) form**

$$p_{\theta s} = f_{\theta t s}\,\Delta t \qquad q_{\theta s} = \mathcal F^{-1}_s\!\big[H(\nu)\,\mathcal F_s[p_{\theta s}]\big] \qquad f_{yx} = Q_{\theta yx}\,\Delta\theta$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $f_{yx}$ | `f` | (N, N) | attenuation map (modified Shepp–Logan: sum of 10 ellipses) |
| $f_{\theta t s}$ | `grid_sample(f, rotated_points(theta, s, t))` | (A, T, S) | image sampled along the ray at distance $s$, position $t$, angle $\theta$ |
| $p_{\theta s}$ | `einsum('ats->as', samples) * dt` | (A, S) | repeated $t$ ⇒ summed: the sinogram |
| $H(\nu)$ | `ramp_filter(S, ds, window)` | (P,) | ramp $/\nu/$ (Ram-Lak), optionally times $\mathrm{sinc}(\nu\,\Delta s)$ (Shepp–Logan), on a zero-padded detector |
| $q_{\theta s}$ | `ifft(fft(p, n=P) * H)` | (A, S) | filtered projections |
| $Q_{\theta yx}$ | `smear(q, theta, x, y)` | (A, N, N) | each projection spread back along its rays: $q_\theta(x\cos\theta + y\sin\theta)$ |
| $Q_{\theta yx}\,\Delta\theta$ | `einsum('ayx,a->yx', Q, dtheta)` | (N, N) | repeated $\theta$ ⇒ summed: the back-projection |

Every projection conserves the total mass, $\int p\,ds = \iint f\,dA$, for each angle. Smearing the raw projections back (plain back-projection) turns a point into a $1/r$ halo, so the image is a blurred version of $f$; the ramp filter $|\nu|$ exactly undoes this blur (central-slice theorem). Too few angles give streaks; the error falls as the number of angles grows until the pixel grid limits it. The Shepp–Logan window softens the ramp at high frequency, trading a little sharpness for less noise — try it with noise switched on. Simplifications: parallel-beam geometry, monochromatic rays, bilinear interpolation, Gaussian detector noise instead of photon statistics. Educational model, not for clinical use.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/radon_ct/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    torch.manual_seed(0)
    x = tensors.grid(params["n_pixels"], dev)  # (N,) pixel centres = detector bins
    f = tensors.shepp_logan(x, tensors.ellipse_params(dev))  # (N, N)
    theta = tensors.angles(params["n_angles"], dev)  # (A,)

    sino = solve.scan(f, theta, x)  # (A, S)
    sino = sino + params["noise"] * sino.max() * torch.randn_like(sino)  # optional detector noise
    plain, filtered = solve.per_angle_backprojections(sino, theta, x, params["filter"])  # (A, N, N) each
    recon = equation.backproject(filtered, theta)

    counts = [8, 16, 32, 64, 128, 256]
    errors = solve.error_vs_angles(f, x, counts, params["filter"])

    dx = float(x[1] - x[0])
    mass = (sino.sum(1) * dx).cpu()
    print(f"image mass ∫f dA = {float(f.sum()) * dx * dx:.4f};  ∫p ds per angle: {mass.min():.4f} … {mass.max():.4f}")
    print(f"sinogram {tuple(sino.shape)}, filter {params['filter']}, noise {params['noise']:.3f}")
    print(f"NRMSE of the FBP reconstruction with {len(theta)} angles: {solve.nrmse(recon, f):.3f}")
    for a, e in zip(counts, errors):
        print(f"  {a:4d} angles  NRMSE {e:.3f}")
    return plot.animate(
        f, x, theta, sino, plain, filtered, counts, errors, solve.nrmse(recon, f), out_dir / "radon_ct.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/radon_ct/equation.py
r"""Physics → tensor translation.

A CT scanner measures line integrals of the attenuation map f(x, y) — the Radon transform.
For a ray at angle θ and signed distance s from the centre, parametrised by t along the ray,

    p(\theta, s) = \int f(s\cos\theta - t\sin\theta,\; s\sin\theta + t\cos\theta)\, dt.

On the grid: sample the image at rotated points (θ, t, s), then sum over t:

    p_{\theta s} = f_{\theta t s}\, \Delta t                    (t summed)

Filtered back-projection inverts it. Filter every projection with the ramp |ν| along s
(a product in Fourier space), then smear each filtered projection back along its rays and
add up all angles:

    q_{\theta s} = \mathcal F^{-1}_s\big[\,|\nu|\, \mathcal F_s[p_{\theta s}]\,\big]
    f_{yx} = Q_{\theta y x}\, \Delta\theta,   Q_{\theta yx} = q_\theta(x\cos\theta + y\sin\theta)   (θ summed)

Coordinates live in [-1, 1]²; pixel and detector centres coincide at θ = 0.
"""

import torch
import torch.nn.functional as F


def rotated_points(theta: torch.Tensor, s: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Sample points of every ray: x = s cosθ − t sinθ, y = s sinθ + t cosθ.  → (A, T, S, 2) as (x, y)."""
    c, sn = torch.cos(theta)[:, None, None], torch.sin(theta)[:, None, None]  # (A, 1, 1)
    x = s[None, None, :] * c - t[None, :, None] * sn
    y = s[None, None, :] * sn + t[None, :, None] * c
    return torch.stack([x, y], dim=-1)


def radon(f: torch.Tensor, theta: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    r"""p_{\theta s} = f_{\theta t s} \Delta t.   f: (N, N) indexed [y, x] → sinogram (A, S).

    grid_sample interpolates f bilinearly at the rotated points; outside [-1, 1]² it returns 0.
    """
    points = rotated_points(theta, s, s)  # use the detector grid for t too: (A, T, S, 2)
    image = f[None, None].expand(len(theta), 1, *f.shape)  # (A, 1, N, N)
    samples = F.grid_sample(image, points, mode="bilinear", align_corners=False)[:, 0]  # (A, T, S)
    dt = s[1] - s[0]
    return torch.einsum("ats->as", samples) * dt  # sum over t


def ramp_filter(n_det: int, ds: float, window: str = "ram-lak", device=None) -> tuple[torch.Tensor, int]:
    r"""Frequency response H(ν) ≈ |ν| on a zero-padded detector of P ≥ 2S bins.

    Built as the FFT of the band-limited ramp kernel h(0) = 1/(4Δs²), h(n odd) = −1/(πnΔs)²
    (Kak & Slaney), which gets the zero-frequency term right. "shepp-logan" multiplies by sinc(ν Δs).
    Returns (H (P,), P).
    """
    pad = 1 << (2 * n_det - 1).bit_length()  # next power of two ≥ 2S: no wrap-around
    idx = torch.arange(pad, device=device)
    n = torch.where(idx < pad // 2, idx, idx - pad).float()
    h = torch.where(n % 2 == 1, -1.0 / (torch.pi * n * ds) ** 2, torch.zeros_like(n))
    h[0] = 1.0 / (4 * ds**2)
    H = ds * torch.fft.fft(h).real  # ≈ |ν| in cycles per unit length
    if window == "shepp-logan":
        nu = torch.fft.fftfreq(pad, d=ds, device=device)
        H = H * torch.sinc(nu * ds)
    return H, pad


def filter_projections(p: torch.Tensor, H: torch.Tensor, pad: int) -> torch.Tensor:
    r"""q_{\theta s} = F^{-1}[H(ν) F[p_θ](ν)] along s, for every angle at once: (A, S) → (A, S)."""
    spectrum = torch.fft.fft(p, n=pad, dim=-1)  # zero-padded FFT along s
    return torch.fft.ifft(spectrum * H, dim=-1).real[:, : p.shape[-1]]


def smear(q: torch.Tensor, theta: torch.Tensor, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    r"""Q_{\theta yx} = q_\theta(x\cos\theta + y\sin\theta): each projection spread back along its rays.

    q: (A, S) on the detector grid (= pixel grid, so s is already a normalised coordinate) → (A, Y, X).
    """
    s = x[None, None, :] * torch.cos(theta)[:, None, None] + y[None, :, None] * torch.sin(theta)[:, None, None]
    points = torch.stack([s, torch.zeros_like(s)], dim=-1)  # (A, Y, X, 2): sample a 1-pixel-high "image"
    return F.grid_sample(q[:, None, None, :], points, mode="bilinear", align_corners=False)[:, 0]


def backproject(Q: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    r"""f_{yx} = Q_{\theta yx} \Delta\theta  (θ summed over [0, π)).   (A, Y, X) → (Y, X)"""
    dtheta = torch.full_like(theta, torch.pi / len(theta))
    return torch.einsum("ayx,a->yx", Q, dtheta)


def ellipse_projection(theta, s, amp, a, b, x0, y0, phi):
    r"""Analytic Radon transform of a uniform ellipse: 2 A a b \sqrt{r^2 - s'^2} / r^2 (0 outside)."""
    s_shift = s[None, :] - x0 * torch.cos(theta)[:, None] - y0 * torch.sin(theta)[:, None]
    r2 = (a * torch.cos(theta - phi)) ** 2 + (b * torch.sin(theta - phi)) ** 2  # (A,)
    inside = (r2[:, None] - s_shift**2).clamp_min(0.0)
    return 2 * amp * a * b * torch.sqrt(inside) / r2[:, None]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/radon_ct/tensors.py
"""Build the concrete tensors: pixel/detector grid, the Shepp–Logan phantom and the projection angles."""

import math

import torch

# Modified Shepp–Logan head phantom (Toft): amplitude A, semi-axes a, b, centre (x0, y0), tilt φ in degrees.
SHEPP_LOGAN = [
    (1.0, 0.69, 0.92, 0.0, 0.0, 0.0),  # skull
    (-0.8, 0.6624, 0.874, 0.0, -0.0184, 0.0),  # brain
    (-0.2, 0.11, 0.31, 0.22, 0.0, -18.0),  # ventricles
    (-0.2, 0.16, 0.41, -0.22, 0.0, 18.0),
    (0.1, 0.21, 0.25, 0.0, 0.35, 0.0),
    (0.1, 0.046, 0.046, 0.0, 0.1, 0.0),
    (0.1, 0.046, 0.046, 0.0, -0.1, 0.0),
    (0.1, 0.046, 0.023, -0.08, -0.605, 0.0),  # three small tumours
    (0.1, 0.023, 0.023, 0.0, -0.606, 0.0),
    (0.1, 0.023, 0.046, 0.06, -0.605, 0.0),
]


def grid(n: int, device: torch.device) -> torch.Tensor:
    """Pixel centres in [-1, 1]: (N,). The detector bins s use the same grid."""
    return (torch.arange(n, device=device) + 0.5) * (2.0 / n) - 1.0


def ellipse_params(device: torch.device) -> torch.Tensor:
    """The phantom's ellipses as a (K, 6) tensor, angles converted to radians."""
    e = torch.tensor(SHEPP_LOGAN, device=device)
    e[:, 5] = torch.deg2rad(e[:, 5])
    return e


def shepp_logan(x: torch.Tensor, ellipses: torch.Tensor) -> torch.Tensor:
    """f_{yx} = A_k · inside_{kyx}  (k summed): the sum of uniform ellipses.  → (Y, X), row index = y."""
    yy, xx = torch.meshgrid(x, x, indexing="ij")  # (Y, X)
    amp, a, b, x0, y0, phi = ellipses.T  # each (K,)
    dx = xx[None] - x0[:, None, None]
    dy = yy[None] - y0[:, None, None]
    c, s = torch.cos(phi)[:, None, None], torch.sin(phi)[:, None, None]
    u, v = dx * c + dy * s, -dx * s + dy * c  # coordinates in each ellipse's own frame
    inside = ((u / a[:, None, None]) ** 2 + (v / b[:, None, None]) ** 2 <= 1.0).float()  # (K, Y, X)
    return torch.einsum("k,kyx->yx", amp, inside)


def disk(x: torch.Tensor, radius: float) -> torch.Tensor:
    """A centred uniform disk of value 1: (Y, X)."""
    yy, xx = torch.meshgrid(x, x, indexing="ij")
    return (xx**2 + yy**2 <= radius**2).float()


def angles(n: int, device: torch.device) -> torch.Tensor:
    """θ_a = π a / A for a = 0 … A−1: a half turn is enough for parallel beams: (A,)."""
    return torch.arange(n, device=device) * (math.pi / n)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/radon_ct/solve.py
"""Scan (sinogram) and reconstruct (plain and filtered back-projection), plus the error vs number of angles."""

import torch

from problems._common.progress import steps

from . import equation, tensors


def scan(f: torch.Tensor, theta: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Sinogram p(θ, s) of the image f: (A, S)."""
    return equation.radon(f, theta, x)


def per_angle_backprojections(p: torch.Tensor, theta: torch.Tensor, x: torch.Tensor, window: str):
    """Smeared projections for each angle, without and with the ramp filter: two (A, Y, X) tensors.

    Summing them over θ (equation.backproject) gives the plain and the filtered reconstruction;
    a running sum shows the image forming while the gantry turns.
    """
    H, pad = equation.ramp_filter(p.shape[-1], float(x[1] - x[0]), window, device=p.device)
    q = equation.filter_projections(p, H, pad)  # (A, S)
    return equation.smear(p, theta, x, x), equation.smear(q, theta, x, x)


def fbp(p: torch.Tensor, theta: torch.Tensor, x: torch.Tensor, window: str = "ram-lak") -> torch.Tensor:
    """Filtered back-projection of a sinogram: (A, S) → (Y, X)."""
    _, filtered = per_angle_backprojections(p, theta, x, window)
    return equation.backproject(filtered, theta)


def nrmse(estimate: torch.Tensor, truth: torch.Tensor) -> float:
    """Normalised root-mean-square error ‖f̂ − f‖ / ‖f‖."""
    return float(torch.linalg.norm(estimate - truth) / torch.linalg.norm(truth))


def error_vs_angles(f: torch.Tensor, x: torch.Tensor, counts: list[int], window: str) -> list[float]:
    """Reconstruct with A evenly spread angles for every A in counts; return the NRMSE of each."""
    errors = []
    for i in steps(len(counts), "angle counts"):
        theta = tensors.angles(counts[i], x.device)
        errors.append(nrmse(fbp(scan(f, theta, x), theta, x, window), f))
    return errors


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/radon_ct/plot.py
"""Animate the scan: the sinogram fills angle by angle while plain and filtered back-projections form."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    f: torch.Tensor,  # (N, N) true phantom
    x: torch.Tensor,  # (N,)
    theta: torch.Tensor,  # (A,)
    sino: torch.Tensor,  # (A, S)
    plain: torch.Tensor,  # (A, N, N) smeared, unfiltered projections
    filtered: torch.Tensor,  # (A, N, N) smeared, ramp-filtered projections
    counts: list[int],
    errors: list[float],
    final_error: float,
    path: Path,
) -> Path:
    f, x, theta, sino = f.cpu().numpy(), x.cpu().numpy(), theta.cpu().numpy(), sino.cpu().numpy()
    dtheta = np.pi / len(theta)
    plain = np.cumsum(plain.cpu().numpy(), axis=0) * dtheta  # running sums over the angles seen so far
    filtered = np.cumsum(filtered.cpu().numpy(), axis=0) * dtheta
    row = int(np.argmin(np.abs(x - 0.0)))  # horizontal profile through the ventricles
    gray = plt.get_cmap("gray", 32)
    ext = (-1, 1, -1, 1)

    fig, axes = plt.subplots(2, 3, figsize=(8.4, 5.0))
    ax_ph, ax_sino, ax_err = axes[0]
    ax_bp, ax_fbp, ax_prof = axes[1]
    for ax in (ax_ph, ax_bp, ax_fbp):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)

    ax_ph.imshow(f, cmap=gray, vmin=0, vmax=0.5, origin="lower", extent=ext)
    ax_ph.set_title("phantom $f(x, y)$")
    ax_ph.set_xlim(-1, 1)
    ax_ph.set_ylim(-1, 1)
    offsets = np.linspace(-0.8, 0.8, 7)
    ray_lines = [ax_ph.plot([], [], color=ACCENT[1], lw=0.8, alpha=0.8)[0] for _ in offsets]
    (detector,) = ax_ph.plot([], [], color=ACCENT[0], lw=3)

    shown = np.full_like(sino.T, np.nan)  # (S, A), filled column by column
    sino_img = ax_sino.imshow(
        shown,
        cmap=plt.get_cmap("viridis", 32),
        vmin=0,
        vmax=sino.max(),
        origin="lower",
        aspect="auto",
        extent=(0, 180, -1, 1),
    )
    ax_sino.set_xlabel(r"$\theta$ (deg)")
    ax_sino.set_ylabel("$s$")
    ax_sino.set_title(r"sinogram $p(\theta, s)$")
    ax_sino.grid(False)

    ax_err.loglog(counts, errors, "o-", color=ACCENT[0], ms=4)
    ax_err.plot([len(theta)], [final_error], "*", color="#f59e0b", ms=11, label=f"this scan ({len(theta)})")
    ax_err.legend(fontsize=7)
    ax_err.yaxis.set_minor_formatter(plt.NullFormatter())
    ax_err.set_yticks([0.25, 0.5, 1.0], ["0.25", "0.5", "1"])
    ax_err.set_xlabel("number of angles")
    ax_err.set_ylabel("NRMSE of FBP")
    ax_err.set_title("error vs angles")

    bp_img = ax_bp.imshow(plain[-1], cmap=gray, origin="lower", extent=ext)
    ax_bp.set_title("plain back-projection")
    fbp_img = ax_fbp.imshow(filtered[-1], cmap=gray, vmin=0, vmax=0.5, origin="lower", extent=ext)
    ax_fbp.set_title("filtered back-projection")

    ax_prof.plot(x, f[row], color="#374151", lw=1.2, label="true")
    (fbp_line,) = ax_prof.plot([], [], color=ACCENT[0], lw=1.2, label="FBP")
    (bp_line,) = ax_prof.plot([], [], color=ACCENT[1], lw=1.2, ls="--", label="plain BP (scaled)")
    ax_prof.set_xlim(-1, 1)
    ax_prof.set_ylim(-0.2, 1.2)
    ax_prof.set_xlabel("$x$ (row $y = 0$)")
    ax_prof.legend(fontsize=7, loc="upper right")
    ax_prof.set_title("profile")

    def update(k: int):
        th = theta[k]
        d = np.array([np.cos(th), np.sin(th)])  # detector axis s
        r = np.array([-np.sin(th), np.cos(th)])  # ray direction t
        for line, s in zip(ray_lines, offsets):
            a, b = s * d - 1.3 * r, s * d + 1.3 * r
            line.set_data([a[0], b[0]], [a[1], b[1]])
        end = -1.0 * r
        detector.set_data([end[0] - 0.9 * d[0], end[0] + 0.9 * d[0]], [end[1] - 0.9 * d[1], end[1] + 0.9 * d[1]])
        shown[:, : k + 1] = sino[: k + 1].T
        sino_img.set_data(shown)
        bp = plain[k]
        bp_img.set_data(bp)
        bp_img.set_clim(0, max(bp.max(), 1e-9))
        fbp_img.set_data(filtered[k])
        fbp_line.set_data(x, filtered[k][row])
        bp_line.set_data(x, bp[row] / max(bp[row].max(), 1e-9))
        ax_fbp.set_title(f"filtered back-projection ({k + 1}/{len(theta)})")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(theta), 48), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_pixels': 128,
    'n_angles': 180,
    'filter': 'ram-lak',
    'noise': 0.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.radon_ct")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.radon_ct.main")
problem_dir = Path("problems/medical/radon_ct")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
